# Gradient term: prototype for the developers

The gradient term corrects each observation's dz and dy for the periscope
gradients OOBAGRD3 and OOBAGRD6. It goes on top of the CALALIGN values.

All of it is in one file, `astrometry_corr_gradient/gradient_term.py`: the code, the numbers
and the test vectors. It needs only numpy and astropy.

This notebook prints that file, then shows:

1. the coefficients,
2. the reference table,
3. the test vectors.

`gradient_term_validation.ipynb` checks that the term works. After a new
export, `python scripts/update_gradient_term_data.py` remakes the numbers.


In [1]:
import inspect
import sys
from pathlib import Path

import numpy as np
from IPython.display import Markdown, display

# this checkout's package, before any installed copy
sys.path.insert(0, str(Path.cwd().parent))
from astrometry_corr_gradient import gradient_term

## The module

The whole file. Its docstring, at the top, explains the term.


In [2]:
print(inspect.getsource(gradient_term))

"""The gradient term: a prototype of what processing needs, for the developers.

The gradient term corrects each observation's position for the periscope
gradient monitors OOBAGRD3 and OOBAGRD6. It goes on top of the CALALIGN
alignment, which is made as today, without the term. For each observation, on
each axis (dz and dy)::

    term = c3 * (OOBAGRD3 - OOBAGRD3 reference) + c6 * (OOBAGRD6 - OOBAGRD6 reference)

The term is the expected X-ray minus catalog offset, in arcsec. Subtract it
from the X-ray position's dz and dy.

The same term, normalized::

    term = k3 * (OOBAGRD3 - OOBAGRD3 reference) / s3 + k6 * (OOBAGRD6 - OOBAGRD6 reference) / s6

- c3, c6: ``COEFFICIENTS``, arcsec per K. Fixed.
- s3, s6: ``SIGMAS``, how much each gradient typically moves inside a calendar
  quarter, in K.
- k3, k6: ``coefficients_per_sigma()``, c x s. Arcsec for that typical move,
  so the two gradients compare directly.
- The references: ``reference_table()``, one pair per calendar quarter.
  Obser

## Coefficients

Two ways to write the same term:

- **Per K:** `term = c3 * (OOBAGRD3 - ref) + c6 * (OOBAGRD6 - ref)`
- **Per sigma:** `term = k3 * (OOBAGRD3 - ref) / s3 + k6 * (OOBAGRD6 - ref) / s6`

**s** is how much each gradient typically moves inside a quarter. **k** = c x s
is the arcsec for that typical move, so the two gradients compare directly.


In [3]:
per_sigma = gradient_term.coefficients_per_sigma()
header = "| axis | gradient | c (arcsec per K) | s (K) | k (arcsec per sigma) |"
rows = [
    f"| {axis_name} | {name} | {gradient_term.COEFFICIENTS[axis_name][name]}"
    f" | {gradient_term.SIGMAS[name]} | {per_sigma[axis_name][name]:.4f} |"
    for axis_name in gradient_term.AXES
    for name in gradient_term.GRADIENT_MSIDS
]
display(Markdown("\n".join([header, "|---|---|---|---|---|", *rows])))
print("COEFFICIENTS =", gradient_term.COEFFICIENTS)
print("SIGMAS =", gradient_term.SIGMAS)
print(
    "per sigma =",
    {
        axis_name: {name: round(k, 4) for name, k in by_gradient.items()}
        for axis_name, by_gradient in per_sigma.items()
    },
)

| axis | gradient | c (arcsec per K) | s (K) | k (arcsec per sigma) |
|---|---|---|---|---|
| dz | OOBAGRD3 | 0.462 | 0.10747 | 0.0497 |
| dz | OOBAGRD6 | -13.284 | 0.00726 | -0.0964 |
| dy | OOBAGRD3 | 1.579 | 0.10747 | 0.1697 |
| dy | OOBAGRD6 | -11.185 | 0.00726 | -0.0812 |

COEFFICIENTS = {'dz': {'OOBAGRD3': 0.462, 'OOBAGRD6': -13.284}, 'dy': {'OOBAGRD3': 1.579, 'OOBAGRD6': -11.185}}
SIGMAS = {'OOBAGRD3': 0.10747, 'OOBAGRD6': 0.00726}
per sigma = {'dz': {'OOBAGRD3': 0.0497, 'OOBAGRD6': -0.0964}, 'dy': {'OOBAGRD3': 0.1697, 'OOBAGRD6': -0.0812}}


## Reference table

One row per calendar quarter. Observations in that quarter use its OOBAGRD3
and OOBAGRD6, in K. `gradient_term()` only looks up the row; how the row was
made does not matter to it. `from_quarter` and `observations` say where each
row's medians come from.


In [4]:
gradient_term.reference_table().pprint(max_lines=-1, max_width=-1)

quarter   start    from_quarter observations OOBAGRD3 OOBAGRD6
                                                K        K    
------- ---------- ------------ ------------ -------- --------
 2004Q1 2004-01-01       2004Q1           47  0.85151  0.04103
 2004Q2 2004-04-01       2004Q2           37  0.85104  0.04103
 2004Q3 2004-07-01       2004Q3           30  0.83347  0.04103
 2004Q4 2004-10-01       2004Q4           48  0.82879  0.04103
 2005Q1 2005-01-01       2005Q1           58  0.83839  0.04107
 2005Q2 2005-04-01       2005Q2           57  0.83153  0.04248
 2005Q3 2005-07-01       2005Q3           36   0.8199  0.04219
 2005Q4 2005-10-01       2005Q4           61  0.81237  0.04169
 2006Q1 2006-01-01       2006Q1           65  0.77691  0.04416
 2006Q2 2006-04-01       2006Q2           59  0.78264  0.04416
 2006Q3 2006-07-01       2006Q3           46  0.77511  0.04416
 2006Q4 2006-10-01       2006Q4           47  0.73879  0.04593
 2007Q1 2007-01-01       2007Q1           52   0.6774  

## Test vectors

Real observations and the term each should get, in arcsec, next to what
`gradient_term()` gives. The last one has no telemetry, so it gets no term.
Any other implementation should match these too.


In [5]:
vectors = gradient_term.vector_table()
term = gradient_term.gradient_term(
    vectors["date"], {name: vectors[name] for name in gradient_term.GRADIENT_MSIDS}
)
for axis_name in gradient_term.AXES:
    vectors[f"{axis_name} computed"] = np.round(term[axis_name], 4)
    vectors[f"{axis_name} computed"].unit = "arcsec"
vectors.pprint(max_width=-1)

obsid         date        OOBAGRD3 OOBAGRD6    dz      dy     no_term    dz computed dy computed
                             K        K      arcsec  arcsec                 arcsec      arcsec  
----- ------------------- -------- -------- ------- ------- ------------ ----------- -----------
 7833 2008-04-01 04:58:27  0.70672  0.05355  0.0128  0.0439            -      0.0128      0.0439
13977 2012-07-01 02:51:34  0.43824  0.06293  0.0966  0.2232            -      0.0966      0.2232
19888 2016-10-01 15:29:43 -0.10007  0.07312  0.1504  0.2662            -      0.1504      0.2662
27455 2022-10-02 05:54:37 -0.51387  0.12805 -0.2286 -0.4378            -     -0.2286     -0.4378
30979 2025-07-02 12:02:23 -0.26719  0.14234 -0.0252  0.0262            -     -0.0252      0.0262
33451 2026-10-03 19:48:41 -0.24373  0.13502 -0.0069 -0.0165            -     -0.0069     -0.0165
11749 2010-05-29 20:41:42      nan      nan     0.0     0.0 no telemetry         0.0         0.0
